# Calibration, Denoising, Baseline Correction, and TIC Normalization

This notebook is configured for the current workspace layout and uses the requested pipeline:
- calibration: prefer `multisegment` with `centroid` + `bootstrap`, and retry with `centroid` + `mz` if the current library hits the known bootstrap fit bug
- denoising: `wavelet`
- baseline correction: `pspline_iarpls(lam=1000000.0)`
- normalization: TIC


In [ ]:
import json
import sys
import warnings
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")


def resolve_notebook_paths():
    cwd = Path.cwd().resolve()
    if (cwd / "NoteBooks").is_dir() and (cwd / "data").is_dir():
        project_root = cwd
        #notebook_root = cwd / "NoteBooks"
        notebook_root = cwd 
        
    #elif (cwd.parent / "data").is_dir():
    project_root = cwd
    notebook_root = cwd
    #else:
    #    raise RuntimeError("Run this notebook from the repo root or from NoteBooks/.")

    output_root = notebook_root / "output_files"
    plots_dir = output_root / "plots"
    output_root.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    return project_root, notebook_root, output_root, plots_dir


PROJECT_ROOT, NOTEBOOK_ROOT, OUTPUT_ROOT, PLOTS_DIR = resolve_notebook_paths()

from mioXpektron import (
    FlexibleCalibrator,
    FlexibleCalibConfig,
    PlotPeak,
    PlotPeaks,
    PlotPeaksConfig,
    BatchDenoising,
    BaselineBatchCorrector,
)


def require_files(pattern):
    matches = sorted(glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files matched pattern: {pattern}")
    return matches


def pick_existing_sample(pattern, preferred="breast_CT-19b_5"):
    matches = require_files(pattern)
    for match in matches:
        if preferred in Path(match).name:
            return Path(match)
    return Path(matches[0])


In [ ]:
data_root = PROJECT_ROOT / "data"

breast_data = require_files(str(data_root / "breast" / "*" / "*.txt"))
blood_data = require_files(str(data_root / "blood" / "*" / "*.txt"))
fur_data = require_files(str(data_root / "fur" / "*" / "*.txt"))
liver_data = require_files(str(data_root / "liver" / "*" / "*.txt"))
all_input_files = sorted(breast_data + blood_data + fur_data + liver_data)

calibrated_dir = OUTPUT_ROOT / "calibrated_spectra"
normalized_output_dir = OUTPUT_ROOT / "normalized_spectra_tic"

print(len(breast_data), "breast files found.")
print(len(blood_data), "blood files found.")
print(len(fur_data), "fur files found.")
print(len(liver_data), "liver files found.")
print(len(all_input_files), "total files found.")
print("Calibration outputs:", calibrated_dir)
print("Normalization outputs:", normalized_output_dir)


In [ ]:
DEFAULT_MASSES = [
    1.0072764666,
    15.0229265168,
    22.9892207021,
    27.0229265168,
    29.0385765812,
    38.9631579065,
    41.0385765812,
    43.0542266457,
    57.0698767102,
    58.065674,
    67.0548,
    71.0855267746,
    86.096976,
    91.0542266457,
    104.107539,
    184.073320,
    224.105171,
    369.351600,
]

CALIBRATION_METHOD = "multisegment"
CALIBRATION_AUTODETECT_METHOD = "centroid"
MAX_ACCEPTABLE_PPM = 100.0
ENABLE_HIGH_ERROR_RESCUE = True
HIGH_ERROR_RETRY_MAX_REMOVALS = 6
AUTODETECT_TOL_DA = (0.75, 0.5, 0.25, 0.1)
MULTISEGMENT_BREAKPOINTS = [50, 150]
 
calibration_attempts = [
    {
        "label": "multisegment_centroid_bootstrap",
        "autodetect_strategy": "bootstrap",
        "prefer_recompute_from_channel": True,
    },
    {
        "label": "multisegment_centroid_mz_fallback",
        "autodetect_strategy": "mz",
        "prefer_recompute_from_channel": False,
    },
]
 
 
last_error = None
for attempt in calibration_attempts:
    try:
        config = FlexibleCalibConfig(
            reference_masses=DEFAULT_MASSES,
            calibration_method=CALIBRATION_METHOD,
            output_folder=str(calibrated_dir),
            autodetect_tol_da=AUTODETECT_TOL_DA,
            autodetect_tol_ppm=None,
            autodetect_method=CALIBRATION_AUTODETECT_METHOD,
            autodetect_strategy=attempt["autodetect_strategy"],
            prefer_recompute_from_channel=attempt["prefer_recompute_from_channel"],
            outlier_threshold=3.0,
            use_outlier_rejection=True,
            max_iterations=5,
            multisegment_breakpoints=MULTISEGMENT_BREAKPOINTS,
            min_calibrants=3,
            max_ppm_threshold=MAX_ACCEPTABLE_PPM,
            fail_on_high_error=True,
            retry_high_error_with_pruning=ENABLE_HIGH_ERROR_RESCUE,
            retry_high_error_with_mz_fallback=ENABLE_HIGH_ERROR_RESCUE,
            retry_high_error_max_removals=HIGH_ERROR_RETRY_MAX_REMOVALS,
            verbose=False,
            max_workers=16,
        )

        calibrator = FlexibleCalibrator(config)
        calibration_summary = calibrator.calibrate(all_input_files, calib_channels_dict=None)
        calibration_mode = attempt["label"]
        break
    except ValueError as exc:
        last_error = exc
        is_bootstrap_bug = (
            attempt["autodetect_strategy"] == "bootstrap"
            and "Initial guess is outside of provided bounds" in str(exc)
        )
        if is_bootstrap_bug:
            print("Bootstrap multisegment fit failed in the current library; retrying with centroid/mz.")
            continue
        raise
else:
    raise last_error

print("Calibration mode:", calibration_mode)
print("Wrote calibrated spectra to", calibrated_dir)

calibration_summary_view = calibration_summary.copy()
calibration_summary_view["ppm_error"] = calibration_summary_view["ppm_error"].round(2)
if "rescue_initial_ppm" in calibration_summary_view.columns:
    calibration_summary_view["rescue_initial_ppm"] = calibration_summary_view["rescue_initial_ppm"].round(2)

rescued_mask = pd.Series(False, index=calibration_summary_view.index)
if "rescue_strategy" in calibration_summary_view.columns:
    rescued_mask = calibration_summary_view["rescue_strategy"].fillna("").astype(str).ne("")

calibration_overview = pd.DataFrame(
    [
        {"metric": "files_calibrated", "value": int(len(calibration_summary_view))},
        {"metric": "mean_ppm_error", "value": round(float(calibration_summary["ppm_error"].mean()), 2)},
        {"metric": "median_ppm_error", "value": round(float(calibration_summary["ppm_error"].median()), 2)},
        {"metric": "max_ppm_error", "value": round(float(calibration_summary["ppm_error"].max()), 2)},
        {"metric": "rescued_files", "value": int(rescued_mask.sum())},
    ]
)

summary_columns = ["file_name", "ppm_error", "n_calibrants"]
for optional_col in ["rescue_strategy", "rescue_initial_ppm", "segment_metrics"]:
    if optional_col in calibration_summary_view.columns:
        summary_columns.append(optional_col)

worst_calibration_results = (
    calibration_summary_view[summary_columns]
    .sort_values("ppm_error", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

display(calibration_overview)
display(worst_calibration_results)

if "rescue_strategy" in calibration_summary_view.columns:
    rescue_summary = (
        calibration_summary_view.assign(
            rescue_strategy=calibration_summary_view["rescue_strategy"].fillna("").replace("", "none")
        )["rescue_strategy"]
        .value_counts()
        .rename_axis("rescue_strategy")
        .reset_index(name="n_files")
    )
    display(rescue_summary)

def _parse_segment_metrics(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, str) and not value.strip():
        return []
    try:
        return json.loads(value)
    except Exception:
        return []

def _format_calibrant_masses(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return ", ".join(f"{float(v):.4f}" for v in value)
    return value

multisegment_detail_rows = []
for _, row in calibration_summary_view.iterrows():
    detail_row = {
        "file_name": row["file_name"],
        "method": row.get("method", ""),
        "ppm_error": row["ppm_error"],
        "n_calibrants": row["n_calibrants"],
        "calibrant_mz_used": _format_calibrant_masses(row.get("calibrant_masses", [])),
    }
    if "rescue_strategy" in calibration_summary_view.columns:
        detail_row["rescue_strategy"] = row.get("rescue_strategy", "")
    if "rescue_initial_ppm" in calibration_summary_view.columns:
        detail_row["rescue_initial_ppm"] = row.get("rescue_initial_ppm", np.nan)

    segment_metrics = _parse_segment_metrics(row.get("segment_metrics", []))
    for metric in segment_metrics:
        segment_name = str(metric.get("segment", "segment"))
        detail_row[f"{segment_name}_ppm"] = (
            None if metric.get("ppm_error") is None else round(float(metric["ppm_error"]), 2)
        )
        detail_row[f"{segment_name}_n"] = metric.get("n_calibrants")
        detail_row[f"{segment_name}_status"] = metric.get("status", "")

    multisegment_detail_rows.append(detail_row)

multisegment_detailed_results = (
    pd.DataFrame(multisegment_detail_rows)
    .sort_values(["ppm_error", "file_name"], ascending=[False, True])
    .reset_index(drop=True)
)

display(multisegment_detailed_results)


In [ ]:
# Configure the plotter
config = PlotPeaksConfig(
    data_dir="data/breast/*",
    file_pattern="*.txt",
    mz_min=40.90,
    mz_max=41.10,
    norm_tic=True,
    save_fig=True  # Set to True to save PDFs to ../outputfiles/plots
)

# Create plotter, load data, and plot
plotter = PlotPeaks(config)
plotter.load_data()
plotter.plot_all()

In [ ]:
# Configure the plotter
config = PlotPeaksConfig(
    data_dir="data/breast/*",
    file_pattern="*.txt",
    mz_min=0,
    mz_max=800,
    norm_tic=True,
    save_fig=True  # Set to True to save PDFs to ../outputfiles/plots
)

# Create plotter, load data, and plot
plotter = PlotPeaks(config)
plotter.load_data()
plotter.plot_all()

In [ ]:
# Configure the plotter
config = PlotPeaksConfig(
    data_dir="data/breast/*",
    file_pattern="*.txt",
    mz_min=50,
    mz_max=250,
    norm_tic=True,
    save_fig=True  # Set to True to save PDFs to ../outputfiles/plots
)

# Create plotter, load data, and plot
plotter = PlotPeaks(config)
plotter.load_data()
plotter.plot_all()

In [ ]:
for tissue in ["breast"]:
    config = PlotPeaksConfig(
        data_dir=str(calibrated_dir),
        file_pattern=f"{tissue}_*.txt",
        mz_min=40.90,
        mz_max=41.10,
        norm_tic=True,
        save_fig=True,
        save_path=str(PLOTS_DIR),
    )
    plotter = PlotPeaks(config)
    plotter.load_data()
    plotter.plot_all()

In [ ]:
for tissue in ["breast", "blood serum", "liver", "Fur"]:
    config = PlotPeaksConfig(
        data_dir=str(calibrated_dir),
        file_pattern=f"{tissue}_*.txt",
        mz_min=40.90,
        mz_max=41.10,
        norm_tic=True,
        save_fig=True,
        save_path=str(PLOTS_DIR),
    )
    plotter = PlotPeaks(config)
    plotter.load_data()
    plotter.plot_all()


In [ ]:
print(glob)
print(type(glob))

In [ ]:
from glob import glob

In [ ]:
files = require_files(str(calibrated_dir / "*.txt"))

WAVELET_PARAMS = {
    "wavelet": "db4",
    "threshold_strategy": "bayes",
    "cycle_spins": 16,
    "sigma_strategy": "global",
    "variance_stabilize": "anscombe",
    "threshold_mode": "soft",
    "pywt_mode": "periodization",
    "clip_nonnegative": True,
    "preserve_tic": False,
}

bd = BatchDenoising(
    file_paths=files,
    method="wavelet",
    n_workers=16,
    backend="threads",
    progress=True,
    params=WAVELET_PARAMS,
)
denoised_summary = bd.run(
    output_root=str(OUTPUT_ROOT),
    folder_name="denoised_spectrums",
    save_result=True,
)
denoised_output_dir = Path(bd.last_output_dir)

print(f"Denoised {len(denoised_summary)} files")
print("Denoised outputs:", denoised_output_dir)


In [ ]:
pre_spec = pick_existing_sample(str(calibrated_dir / "breast_CT-19b_5_calibrated.txt"))
post_spec = denoised_output_dir / f"{pre_spec.stem}_denoised.txt"

if not post_spec.exists():
    raise FileNotFoundError(post_spec)

data_pre = pd.read_csv(pre_spec, sep="\t", comment="#")
data_post = pd.read_csv(post_spec, sep="\t", comment="#")

pp = PlotPeak(
    mz_values=data_pre["m/z"].values,
    raw_intensities=data_pre["Intensity"].values,
    corrected_intensities=data_post["Intensity"].values,
    sample_name=pre_spec.stem,
)
pp.plot(
    show_peaks=False,
    mz_min=117.5,
    mz_max=118.5,
    figsize=(10, 6),
    save_plot=True,
)


In [ ]:
baseline_method = "pspline_iarpls"
baseline_kwargs = {"lam": 1000000.0}

bbc = BaselineBatchCorrector(
    in_dir=str(denoised_output_dir),
    pattern="*.txt",
    method=baseline_method,
    method_kwargs=baseline_kwargs,
    n_jobs=12,
)
baseline_output_dir = Path(bbc.run(out_root=str(OUTPUT_ROOT)))

print("Baseline outputs:", baseline_output_dir)


In [ ]:
denoised_spec = pick_existing_sample(str(denoised_output_dir / "breast_CT-19b_5_calibrated_denoised.txt"))
baseline_spec = baseline_output_dir / f"{denoised_spec.stem}_baseline_corrected.csv"

if not baseline_spec.exists():
    raise FileNotFoundError(baseline_spec)

data_pre = pd.read_csv(denoised_spec, sep="\t", comment="#")
data_post = pd.read_csv(baseline_spec)

pp = PlotPeak(
    mz_values=data_pre["m/z"].values,
    raw_intensities=np.log1p(data_pre["Intensity"].values),
    corrected_intensities=np.log1p(data_post["corrected_intensity"].values),
    sample_name=denoised_spec.stem,
)
pp.plot(
    show_peaks=False,
    mz_min=0,
    mz_max=None,
    figsize=(10, 6),
    save_plot=False,
)


In [ ]:
normalized_output_dir.mkdir(parents=True, exist_ok=True)
normalized_files = []

for input_file in require_files(str(baseline_output_dir / "*.csv")):
    df = pd.read_csv(input_file)
    intensity_col = "corrected_intensity" if "corrected_intensity" in df.columns else "intensity"
    channel = df["channel"] if "channel" in df.columns else pd.Series(range(len(df)))
    tic = df[intensity_col].sum()
    if tic == 0:
        normalized = df[intensity_col].astype(float)
    else:
        normalized = df[intensity_col].astype(float) * (1000000.0 / float(tic))

    source_col = f"source_{intensity_col}"
    output_df = pd.DataFrame({
        "channel": channel,
        "mz": df["mz"],
        source_col: df[intensity_col],
        "intensity": normalized,
    })
    output_path = normalized_output_dir / f"{Path(input_file).stem}_normalized.csv"
    output_df.to_csv(output_path, index=False)
    normalized_files.append(str(output_path))

print(f"Normalized {len(normalized_files)} files")
print("Normalized outputs:", normalized_output_dir)


In [ ]:
print(glob)
print(type(glob))

In [ ]:
baseline_spec = pick_existing_sample(str(baseline_output_dir / "breast_CT-19b_5*_baseline_corrected.csv"))
normalized_spec = normalized_output_dir / f"{baseline_spec.stem}_normalized.csv"

if not normalized_spec.exists():
    raise FileNotFoundError(normalized_spec)

data_pre = pd.read_csv(baseline_spec)
data_post = pd.read_csv(normalized_spec)

pp = PlotPeak(
    mz_values=data_pre["mz"].values,
    raw_intensities=np.log1p(data_pre["corrected_intensity"].values),
    corrected_intensities=np.log1p(data_post["intensity"].values),
    sample_name=baseline_spec.stem,
)
pp.plot(
    show_peaks=False,
    mz_min=0,
    mz_max=None,
    figsize=(5, 3),
    save_plot=True,
)

In [ ]:
config = PlotPeaksConfig(
    data_dir=str(normalized_output_dir),
    file_pattern="*.csv",
    mz_min=50,
    mz_max=41.10,
    norm_tic=False,
    save_fig=False,
    save_path=str(PLOTS_DIR),
)

plotter = PlotPeaks(config)
plotter.load_data()
plotter.plot_all()


In [ ]:
PlotPeaksConfig

In [ ]:
for tissue in ["breast"]:
    config = PlotPeaksConfig(
        data_dir=str(normalized_output_dir),
        file_pattern="*.csv",
        mz_min=0,
        mz_max=800,
        norm_tic=True,
        save_fig=True,
        save_path=str(PLOTS_DIR),
    )
    plotter = PlotPeaks(config)
    plotter.load_data()
    plotter.plot_all()

In [ ]:
print("Calibration:", calibrated_dir)
print("Denoising:", denoised_output_dir)
print("Baseline correction:", baseline_output_dir)
print("TIC normalization:", normalized_output_dir)


# To export

In [ ]:
# export peak to .csv file with specific range

import glob
from mioXpektron import PeakAlignIntensityArea

input_files = 'output_files/normalized_spectra_tic/*.csv'
files = glob.glob(input_files)
output_files_path = 'output_files/peak_analysis/all4_50-250'

pa = PeakAlignIntensityArea(
    mz_tolerance=0.2,
    mz_rounding_precision=2,
    min_snr=3,
    prominence=5,
    output_dir=output_files_path,
    verbose=True
)

intensity_table, area_table, peaks_df = pa.run(
    csv_files=files,
    mz_min=50,
    mz_max=250,
)

In [ ]:
# ----------from csv to txt-------with grouping samples-------

import pandas as pd
import re

# ---------- Paths ----------
input_csv = "output_files/peak_analysis/all4_50-250/peak_area_table.csv"
output_txt = "output_files/peak_analysis/all4_50-250/peak_area_table_new.txt"

# ---------- Load CSV ----------
df = pd.read_csv(input_csv)

# ---------- Parse SampleName ----------
def parse_sample(sample):
    core = sample.split("_calibrated")[0]

    # Allow tissue names like:
    # Fur
    # blood serum
    # breast
    # liver
    match = re.match(r"(.+?)_(CC|CT)-([0-9]+[a-z])_([0-9]+)$", core)
    if not match:
        raise ValueError(f"Unexpected SampleName format: {sample}")

    tissue_raw, ccct, mouse, spot = match.groups()

    tissue_map = {
        "breast": "Breast tissue",
        "blood serum": "Blood serum",
        "liver": "Liver",
        "Fur": "Fur",
    }

    tissue = tissue_map.get(tissue_raw, tissue_raw)
    index = f"{tissue_raw}_{ccct}-{mouse}_{spot}"
    group = "Cancer" if ccct == "CC" else "Control"

    return pd.Series([index, tissue, mouse, spot, group])

# ---------- Apply ----------
df[["Index", "Tissue", "Mouse", "Spot", "Group"]] = df["SampleName"].apply(parse_sample)

# ---------- Reorder columns ----------
new_order = ["Index", "Tissue", "Mouse", "Spot", "Group"] + \
            [c for c in df.columns if c not in ["SampleName", "Index", "Tissue", "Mouse", "Spot", "Group"]]

df = df[new_order]

# ---------- Save as TAB-delimited TXT ----------
df.to_csv(output_txt, sep="\t", index=False)

print(f"Saved: {output_txt}")
